# FIT5230 Milestone 1 - Context Matters

**Context-Conditioned Lip Dynamics for Speech-to-Face Deepfake Detection**

**Theme 3: Speech-to-Face | Side: Light (Defense)**

This is our initial customized pipeline for **context-conditioned lip-dynamics analysis of talking-face videos**. It accepts short real and generated MP4 clips, extracts normalized lip landmarks, derives temporal motion features, visualizes the trajectories, and exports reproducible results.

Our research hypothesis is that the visible realization of a phoneme depends on its neighboring phonetic context (coarticulation), and that some S2F generators may reproduce this context-conditioned motion differently from real speakers. This notebook intentionally begins with a lightweight, interpretable pilot before adding automatic phoneme alignment or a large classifier. Use only consented or appropriately licensed media.

## 0. Research basis and reproducibility information

- **Main technical baseline:** [PIA](https://github.com/skrantidatta/PIA)
- **Main paper:** [PIA: Deepfake Detection Using Phoneme-Temporal and Identity-Dynamic Analysis](https://openaccess.thecvf.com/content/ICCV2025W/APAI/papers/Datta_PIA_Deepfake_Detection_Using_Phoneme-Temporal_and_Identity-Dynamic_Analysis_ICCVW_2025_paper.pdf)
- **Theoretical motivation:** [CALS](https://arxiv.org/abs/2305.19556)
- **General video-detection reference:** [GenVidBench](https://github.com/genvidbench/GenVidBench)

PIA inference and training are not claimed as completed in this small Milestone 1 notebook. Instead, the notebook provides our own S2F-specific preprocessing and lip-dynamics customization. Later milestones will evaluate PIA, a self-contained temporal baseline, and our context-conditioned extension under speaker-disjoint and generator-disjoint protocols.

In [ ]:
# Run once in Google Colab. If Colab asks for a runtime restart after installation,
# restart the session and continue from the next cell.
%pip -q install "mediapipe==0.10.30" "opencv-python-headless>=4.11,<6" pandas matplotlib


In [ ]:
from pathlib import Path
import json
import math
import platform
import shutil
import subprocess
import sys
import urllib.request
import warnings

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)


## 1. Configure two short input videos

For the first milestone, use one clear frontal real clip and one S2F/lip-synced clip, preferably 3-10 seconds each. The clips do not need to contain the same identity, but a controlled pair using the same audio is better for the later experiment.

Two input options are supported:

1. Upload files to the Colab session and enter their `/content/...` paths below.
2. Mount Google Drive and enter the Drive paths.

The notebook includes a small public-data smoke test so the complete pipeline can be verified without private media. The public real and generated clips are **not a controlled scientific pair**: they have different identities and speech. Their role is only to demonstrate that video loading, landmark tracking, feature extraction, plotting, and export work end to end. Replace them with controlled data before reporting comparative research results.

Public smoke-test sources:

- Real: Wikitongues, *Simon speaking Cumbrian*, CC BY 3.0, attribution: Wikitongues - https://commons.wikimedia.org/wiki/File:WIKITONGUES-_Simon_speaking_Cumbrian.webm
- Generated: TalkingHeadBench, Hallo test sample - https://huggingface.co/datasets/luchaoqi/TalkingHeadBench

The notebook also downloads Google's official MediaPipe Face Landmarker model (about 3.6 MB). Set `USE_PUBLIC_DEMO = False` to use your own clips instead.

In [ ]:
USE_PUBLIC_DEMO = True
VIDEO_PATHS = {
    "real": "",       # Example: /content/real_clip.mp4
    "generated": "",  # Example: /content/generated_clip.mp4
}

FRAME_STRIDE = 2          # Analyze every second frame for a lightweight demo.
SMOOTHING_WINDOW = 5      # Rolling-median window in analyzed frames.
MAX_DURATION_S = 15.0     # Safety cap for the Milestone 1 demo.
OUTPUT_DIR = Path("milestone1_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

PUBLIC_DATA_DIR = Path("public_demo_data")
MODEL_DIR = Path("models")
FACE_LANDMARKER_MODEL = MODEL_DIR / "face_landmarker.task"
FACE_LANDMARKER_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/face_landmarker/"
    "face_landmarker/float16/latest/face_landmarker.task"
)
REAL_SOURCE_URL = (
    "https://commons.wikimedia.org/wiki/Special:Redirect/file/"
    "WIKITONGUES-_Simon_speaking_Cumbrian.webm"
)
FAKE_SOURCE_URL = (
    "https://huggingface.co/datasets/luchaoqi/TalkingHeadBench/resolve/main/"
    "fake/Hallo/test/00023--7lHr2ZNlULA_4--Hallo.mp4?download=true"
)

def download_public_file(url, destination):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        print("Using cached", destination)
        return destination
    request = urllib.request.Request(
        url, headers={"User-Agent": "FIT5230-Milestone1-Academic-Demo/1.0"}
    )
    with urllib.request.urlopen(request, timeout=120) as response, destination.open("wb") as stream:
        shutil.copyfileobj(response, stream)
    print("Downloaded", destination, f"({destination.stat().st_size / 1e6:.2f} MB)")
    return destination

MODEL_DIR.mkdir(exist_ok=True)
download_public_file(FACE_LANDMARKER_MODEL_URL, FACE_LANDMARKER_MODEL)

if USE_PUBLIC_DEMO:
    PUBLIC_DATA_DIR.mkdir(exist_ok=True)
    real_webm = download_public_file(REAL_SOURCE_URL, PUBLIC_DATA_DIR / "wikitongues_real.webm")
    fake_mp4 = download_public_file(FAKE_SOURCE_URL, PUBLIC_DATA_DIR / "talkingheadbench_hallo_fake.mp4")
    real_mp4 = PUBLIC_DATA_DIR / "wikitongues_real_12s.mp4"
    if not real_mp4.exists():
        command = [
            "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
            "-ss", "2", "-i", str(real_webm), "-t", "12",
            "-c:v", "libx264", "-preset", "veryfast", "-crf", "23",
            "-c:a", "aac", "-b:a", "128k", str(real_mp4),
        ]
        subprocess.run(command, check=True)
        print("Created", real_mp4)
    VIDEO_PATHS = {"real": str(real_mp4), "generated": str(fake_mp4)}

# Optional Colab upload helper. Uncomment these lines if needed.
# from google.colab import files
# uploaded = files.upload()
# print("Uploaded:", list(uploaded))

available_paths = {
    label: Path(path) for label, path in VIDEO_PATHS.items() if path and Path(path).exists()
}
if available_paths:
    print("Ready to analyze:", available_paths)
else:
    print("No input videos configured yet. Upload clips and update VIDEO_PATHS.")


## 2. Feature definition

We use the current MediaPipe Face Landmarker Tasks API and normalize all distances by the distance between the outer eye corners. This reduces sensitivity to face size and camera distance.

Initial features:

- `inner_aperture`: distance between the inner upper and lower lips;
- `outer_aperture`: distance between the outer upper and lower lips;
- `lip_width`: distance between left and right mouth corners;
- `velocity`, `acceleration`, and `jerk`: temporal derivatives of smoothed inner aperture;
- `detected`: whether a face was detected in the sampled frame.

These features do not by themselves prove coarticulation. They provide a reproducible first measurement that can later be aligned with phoneme intervals.

In [ ]:
# MediaPipe landmark indices used in this notebook.
LEFT_EYE_OUTER = 33
RIGHT_EYE_OUTER = 263
LEFT_MOUTH_CORNER = 61
RIGHT_MOUTH_CORNER = 291
INNER_UPPER_LIP = 13
INNER_LOWER_LIP = 14
OUTER_UPPER_LIP = 0
OUTER_LOWER_LIP = 17
OUTER_LIP_CONTOUR = [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 409, 270, 269, 0, 37, 39, 40, 185, 61]
INNER_LIP_CONTOUR = [78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308, 415, 310, 311, 13, 82, 81, 80, 191, 78]

def _xy(landmarks, index):
    point = landmarks[index]
    return np.array([point.x, point.y], dtype=np.float64)

def _distance(landmarks, first, second):
    return float(np.linalg.norm(_xy(landmarks, first) - _xy(landmarks, second)))

def get_video_metadata(video_path):
    """Return basic media properties used for input quality control."""
    video_path = Path(video_path)
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise ValueError(f"Could not open video: {video_path}")
    fps = float(capture.get(cv2.CAP_PROP_FPS))
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    capture.release()
    duration_s = frame_count / fps if fps > 0 else np.nan
    return {
        "file": video_path.name,
        "width": width,
        "height": height,
        "fps": fps,
        "frame_count": frame_count,
        "duration_s": duration_s,
    }

def create_face_landmarker(running_mode):
    """Create the current MediaPipe Tasks face-landmark detector on CPU."""
    options = vision.FaceLandmarkerOptions(
        base_options=mp_python.BaseOptions(
            model_asset_path=str(FACE_LANDMARKER_MODEL),
            delegate=mp_python.BaseOptions.Delegate.CPU,
        ),
        running_mode=running_mode,
        num_faces=1,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return vision.FaceLandmarker.create_from_options(options)

def detect_face_landmarks(detector, frame_bgr, timestamp_ms=None):
    """Return one face's landmarks, or None when no face is detected."""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    if timestamp_ms is None:
        result = detector.detect(image)
    else:
        result = detector.detect_for_video(image, int(timestamp_ms))
    return result.face_landmarks[0] if result.face_landmarks else None

def create_landmark_preview(video_path, output_path=None, max_frames=120):
    """Draw the tracked mouth contour on the first confidently detected frame."""
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    preview_bgr = None
    with create_face_landmarker(vision.RunningMode.IMAGE) as detector:
        for _ in range(max_frames):
            ok, frame = capture.read()
            if not ok:
                break
            landmarks = detect_face_landmarks(detector, frame)
            if landmarks is None:
                continue

            preview_bgr = frame.copy()
            height, width = preview_bgr.shape[:2]
            for contour, color in [(OUTER_LIP_CONTOUR, (0, 255, 0)), (INNER_LIP_CONTOUR, (0, 200, 255))]:
                points = [
                    (int(landmarks[index].x * width), int(landmarks[index].y * height))
                    for index in contour
                ]
                for first, second in zip(points[:-1], points[1:]):
                    cv2.line(preview_bgr, first, second, color, 2, cv2.LINE_AA)
                for point in points[:-1]:
                    cv2.circle(preview_bgr, point, 2, color, -1, cv2.LINE_AA)
            cv2.putText(
                preview_bgr, "Tracked lip landmarks", (16, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (40, 220, 40), 2, cv2.LINE_AA
            )
            break

    capture.release()
    if preview_bgr is None:
        raise ValueError("No face was detected for the landmark preview.")
    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(output_path), preview_bgr)
    return cv2.cvtColor(preview_bgr, cv2.COLOR_BGR2RGB)

def extract_lip_dynamics(
    video_path,
    frame_stride=2,
    smoothing_window=5,
    max_duration_s=15.0,
):
    """Extract normalized lip geometry and temporal derivatives from one video."""
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(video_path)
    if frame_stride < 1:
        raise ValueError("frame_stride must be at least 1")

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = float(capture.get(cv2.CAP_PROP_FPS))
    if not np.isfinite(fps) or fps <= 0:
        fps = 25.0

    rows = []
    frame_index = 0

    with create_face_landmarker(vision.RunningMode.VIDEO) as detector:
        while True:
            ok, frame = capture.read()
            if not ok:
                break

            time_s = frame_index / fps
            if max_duration_s is not None and time_s > max_duration_s:
                break

            if frame_index % frame_stride != 0:
                frame_index += 1
                continue

            landmarks = detect_face_landmarks(
                detector, frame, timestamp_ms=round(time_s * 1000)
            )
            row = {
                "frame_index": frame_index,
                "time_s": time_s,
                "detected": False,
                "inner_aperture": np.nan,
                "outer_aperture": np.nan,
                "lip_width": np.nan,
            }

            if landmarks is not None:
                face_scale = _distance(landmarks, LEFT_EYE_OUTER, RIGHT_EYE_OUTER)
                if face_scale > 1e-6:
                    row.update({
                        "detected": True,
                        "inner_aperture": _distance(landmarks, INNER_UPPER_LIP, INNER_LOWER_LIP) / face_scale,
                        "outer_aperture": _distance(landmarks, OUTER_UPPER_LIP, OUTER_LOWER_LIP) / face_scale,
                        "lip_width": _distance(landmarks, LEFT_MOUTH_CORNER, RIGHT_MOUTH_CORNER) / face_scale,
                    })

            rows.append(row)
            frame_index += 1

    capture.release()
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f"No sampled frames were read from {video_path}")
    if not df["detected"].any():
        raise ValueError("No face was detected. Try a shorter, clearer, frontal clip.")

    geometry_columns = ["inner_aperture", "outer_aperture", "lip_width"]
    df[geometry_columns] = df[geometry_columns].interpolate(
        method="linear", limit_direction="both"
    )

    window = max(1, int(smoothing_window))
    for column in geometry_columns:
        df[f"{column}_smooth"] = df[column].rolling(
            window=window, center=True, min_periods=1
        ).median()

    time_values = df["time_s"].to_numpy(dtype=np.float64)
    aperture = df["inner_aperture_smooth"].to_numpy(dtype=np.float64)
    if len(df) >= 3:
        velocity = np.gradient(aperture, time_values)
        acceleration = np.gradient(velocity, time_values)
        jerk = np.gradient(acceleration, time_values)
    else:
        velocity = np.zeros_like(aperture)
        acceleration = np.zeros_like(aperture)
        jerk = np.zeros_like(aperture)

    df["aperture_velocity"] = velocity
    df["aperture_acceleration"] = acceleration
    df["aperture_jerk"] = jerk
    df.attrs["video_path"] = str(video_path)
    df.attrs["fps"] = fps
    df.attrs["frame_stride"] = frame_stride
    return df

def summarize_lip_dynamics(df):
    velocity = df["aperture_velocity"].to_numpy(dtype=np.float64)
    jerk = df["aperture_jerk"].to_numpy(dtype=np.float64)
    return {
        "sampled_frames": int(len(df)),
        "duration_s": float(df["time_s"].iloc[-1] - df["time_s"].iloc[0]),
        "landmark_coverage": float(df["detected"].mean()),
        "mean_inner_aperture": float(df["inner_aperture_smooth"].mean()),
        "std_inner_aperture": float(df["inner_aperture_smooth"].std(ddof=0)),
        "mean_lip_width": float(df["lip_width_smooth"].mean()),
        "mean_abs_velocity": float(np.mean(np.abs(velocity))),
        "p95_abs_jerk": float(np.percentile(np.abs(jerk), 95)),
        "exploratory_freeze_ratio": float(np.mean(np.abs(velocity) < 0.01)),
    }

def plot_lip_dynamics(results, output_path=None):
    if not results:
        print("No extracted results to plot.")
        return

    figure, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
    for label, df in results.items():
        axes[0].plot(df["time_s"], df["inner_aperture_smooth"], label=label)
        axes[1].plot(df["time_s"], df["lip_width_smooth"], label=label)
        axes[2].plot(df["time_s"], df["aperture_velocity"], label=label)

    axes[0].set_ylabel("Inner aperture")
    axes[1].set_ylabel("Lip width")
    axes[2].set_ylabel("Aperture velocity")
    axes[2].set_xlabel("Time (s)")
    axes[0].set_title("Normalized lip-dynamics traces")
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend()
    plt.tight_layout()
    if output_path is not None:
        figure.savefig(output_path, dpi=180, bbox_inches="tight")
        print("Saved", output_path)
    plt.show()
    return figure


## 3. Run the initial extraction

The cell reports landmark coverage for quality control. For this small pilot, replace clips with clearer frontal samples when coverage is low rather than silently treating missing landmarks as a forensic cue.

In [ ]:
metadata_rows = []
preview_images = {}
for label, path in available_paths.items():
    try:
        metadata_rows.append({"label": label, **get_video_metadata(path)})
        preview_path = OUTPUT_DIR / f"landmark_preview_{label}.png"
        preview_images[label] = create_landmark_preview(path, preview_path)
    except Exception as error:
        print(f"Input QC warning for {label}: {error}")

if metadata_rows:
    print("Input media quality-control table")
    display(pd.DataFrame(metadata_rows).set_index("label").round(3))

if preview_images:
    figure, axes = plt.subplots(1, len(preview_images), figsize=(6 * len(preview_images), 5))
    axes = np.atleast_1d(axes)
    for axis, (label, image) in zip(axes, preview_images.items()):
        axis.imshow(image)
        axis.set_title(f"{label.title()} - tracked mouth")
        axis.axis("off")
    figure.tight_layout()
    preview_grid_path = OUTPUT_DIR / "landmark_previews.png"
    figure.savefig(preview_grid_path, dpi=180, bbox_inches="tight")
    print("Saved", preview_grid_path)
    plt.show()

results = {}
summaries = {}

for label, path in available_paths.items():
    print(f"Analyzing {label}: {path}")
    try:
        result_df = extract_lip_dynamics(
            path,
            frame_stride=FRAME_STRIDE,
            smoothing_window=SMOOTHING_WINDOW,
            max_duration_s=MAX_DURATION_S,
        )
        results[label] = result_df
        summaries[label] = summarize_lip_dynamics(result_df)
    except Exception as error:
        print(f"Skipped {label}: {error}")

if summaries:
    display(pd.DataFrame(summaries).T.round(4))
    comparison_path = OUTPUT_DIR / "lip_dynamics_comparison.png"
    plot_lip_dynamics(results, comparison_path)
else:
    print("Extraction has not run yet. Configure at least one valid VIDEO_PATHS entry.")


## 4. Controlled word/phoneme intervals

Automatic forced alignment is deliberately postponed until the lightweight video pipeline is verified. For Milestone 1, manually enter one or two approximate intervals using a media player. Suitable first prompts include `bee`, `boo`, and `bay`, which keep a central bilabial consonant while changing the following vowel.

Manual intervals are pilot annotations, not final ground truth. Later milestones will replace them with automatic alignment plus manual quality checks.

In [ ]:
# Replace these examples after inspecting your own clips.
MANUAL_SEGMENTS = {
    # "real": [
    #     {"label": "bee", "start_s": 0.80, "end_s": 1.25},
    #     {"label": "boo", "start_s": 1.70, "end_s": 2.15},
    # ],
    # "generated": [
    #     {"label": "bee", "start_s": 0.80, "end_s": 1.25},
    #     {"label": "boo", "start_s": 1.70, "end_s": 2.15},
    # ],
}

def summarize_segments(results, segment_map):
    rows = []
    for source_label, segments in segment_map.items():
        if source_label not in results:
            continue
        df = results[source_label]
        for segment in segments:
            start_s = float(segment["start_s"])
            end_s = float(segment["end_s"])
            window = df[(df["time_s"] >= start_s) & (df["time_s"] <= end_s)]
            if window.empty:
                continue
            rows.append({
                "source": source_label,
                "segment": segment["label"],
                "start_s": start_s,
                "end_s": end_s,
                "mean_aperture": float(window["inner_aperture_smooth"].mean()),
                "max_aperture": float(window["inner_aperture_smooth"].max()),
                "mean_width": float(window["lip_width_smooth"].mean()),
                "mean_abs_velocity": float(window["aperture_velocity"].abs().mean()),
                "p95_abs_jerk": float(np.percentile(window["aperture_jerk"].abs(), 95)),
            })
    return pd.DataFrame(rows)

segment_summary = summarize_segments(results, MANUAL_SEGMENTS)
if segment_summary.empty:
    print("No manual segments configured yet. Add timestamps above for the controlled pilot.")
else:
    display(segment_summary.round(4))


## 5. Export reproducible outputs

Each analyzed clip is exported to CSV together with a machine-readable summary. These files provide evidence of our initial customization and will become inputs to the Milestone 2 detector and challenge code.

In [ ]:
for label, df in results.items():
    csv_path = OUTPUT_DIR / f"{label}_lip_dynamics.csv"
    df.to_csv(csv_path, index=False)
    print("Saved", csv_path)

if summaries:
    summary_path = OUTPUT_DIR / "summary.json"
    summary_path.write_text(json.dumps(summaries, indent=2), encoding="utf-8")
    print("Saved", summary_path)

environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "opencv": cv2.__version__,
    "mediapipe": mp.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "frame_stride": FRAME_STRIDE,
    "smoothing_window": SMOOTHING_WINDOW,
    "max_duration_s": MAX_DURATION_S,
}
environment_path = OUTPUT_DIR / "environment.json"
environment_path.write_text(json.dumps(environment, indent=2), encoding="utf-8")
print("Saved", environment_path)

if not segment_summary.empty:
    segment_path = OUTPUT_DIR / "manual_segment_summary.csv"
    segment_summary.to_csv(segment_path, index=False)
    print("Saved", segment_path)

submission_template = pd.DataFrame(columns=[
    "clip_id", "prediction", "fake_probability", "start_s",
    "end_s", "cue", "comment"
])
template_path = OUTPUT_DIR / "challenge_submission_template.csv"
submission_template.to_csv(template_path, index=False)
print("Saved", template_path)


## 6. Interpretation limits and next steps

The current plots are **exploratory**. A difference between two individual clips is not evidence that a feature generalizes to all real or generated videos. We will avoid reporting accuracy until we have identity-separated train/test data.

Planned extensions:

1. add automatic phoneme alignment with manual quality control;
2. construct controlled context-matched hard negatives with the same central phoneme;
3. add a lightweight context-matching model;
4. compare against PIA, a context-free temporal baseline, and general visual evidence;
5. use speaker-disjoint, phrase-disjoint, and generator-disjoint evaluation;
6. test compression, resizing, frame-rate changes, and mild noise; and
7. release a functional challenge notebook for other teams in Milestone 2.

If coarticulation is not sufficiently discriminative, the same pipeline supports a safe course-project fallback using speech-boundary dynamics, motion smoothness, and fusion with the baseline detector.